In [16]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelDecoder

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
# torch.manual_seed(1337)
torch.cuda.manual_seed_all(1337) #gpu
print("device:", device)

device: cuda


In [2]:
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")
texts = [x for x in dataset["train"]["text"] if x.strip()]
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()

trainer = BpeTrainer(
    vocab_size=5000,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"]
)

tokenizer.train_from_iterator(texts, trainer=trainer)


In [3]:
tokenizer.save("tokenizer.json")

In [4]:
# =========================
# 1. 把 texts 编码成一个长序列
# =========================
all_token_ids = []

for i, text in enumerate(texts):
    text = text.strip()
    if not text:
        continue

    ids = tokenizer.encode(text).ids
    all_token_ids.extend(ids)

    if i % 5000 == 0:
        print(f"encoded {i} lines")

data = torch.tensor(all_token_ids, dtype=torch.long)
vocab_size = tokenizer.get_vocab_size()

print("vocab_size:", vocab_size)
print("data_len:", len(data))


# =========================
# 2. 编码 / 解码函数
# =========================
def encode(text: str) -> list[int]:
    return tokenizer.encode(text).ids


def decode(token_ids) -> str:
    return tokenizer.decode([int(i) for i in token_ids])


# =========================
# 3. 保存 tokenizer 和数据
# =========================
tokenizer.save("tokenizer.json")
torch.save(data, "dataset.pt")

print("saved tokenizer.json and dataset.pt")


# =========================
# 4. 训练参数
# =========================
block_size = 128
batch_size = 16

if len(data) <= block_size:
    raise ValueError(
        f"data 太短了，len(data)={len(data)}，但 block_size={block_size}"
    )


# =========================
# 5. 取一个 batch
# =========================
def get_batch():
    start_idx = torch.randint(0, len(data) - block_size - 1, (batch_size,))

    x = torch.stack([data[i : i + block_size] for i in start_idx])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in start_idx])

    return x.to(device), y.to(device)


xb, yb = get_batch()
print("xb:", xb.shape)
print("yb:", yb.shape)


# =========================
# 6. 简单测试 tokenizer
# =========================
sample_text = "To be or not to be"
sample_ids = encode(sample_text)

print("sample_text:", sample_text)
print("encoded:", sample_ids)
print("decoded:", decode(sample_ids))

encoded 0 lines
encoded 5000 lines
encoded 10000 lines
encoded 15000 lines
encoded 20000 lines
encoded 25000 lines
encoded 30000 lines
encoded 35000 lines
encoded 40000 lines
encoded 45000 lines
encoded 50000 lines
encoded 55000 lines
encoded 60000 lines
encoded 65000 lines
encoded 70000 lines
encoded 75000 lines
encoded 80000 lines
encoded 85000 lines
encoded 90000 lines
encoded 95000 lines
encoded 100000 lines
encoded 105000 lines
encoded 110000 lines
encoded 115000 lines
encoded 120000 lines
encoded 125000 lines
encoded 130000 lines
encoded 135000 lines
encoded 140000 lines
encoded 145000 lines
encoded 150000 lines
encoded 155000 lines
encoded 160000 lines
encoded 165000 lines
encoded 170000 lines
encoded 175000 lines
encoded 180000 lines
encoded 185000 lines
encoded 190000 lines
encoded 195000 lines
encoded 200000 lines
encoded 205000 lines
encoded 210000 lines
encoded 215000 lines
encoded 220000 lines
encoded 225000 lines
encoded 230000 lines
encoded 235000 lines
encoded 240000 li

In [23]:
# 读取 tokenizer
tokenizer = Tokenizer.from_file("tokenizer.json")
# 读取数据
data = torch.load("dataset.pt", map_location="cpu")

# 恢复一些常用变量
vocab_size = tokenizer.get_vocab_size()
block_size = 128
batch_size = 16

def encode(text):
    return tokenizer.encode(text).ids

def decode(token_ids):
    # 使用 tokenizer 原生解码，不去管它带不带 Ġ
    text = tokenizer.decode([int(i) for i in token_ids])
    # 手动处理：把 ByteLevel 的空格标记替换回标准空格
    return text.replace("Ġ", " ")

def get_batch(device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# 测试
print("vocab_size:", vocab_size)
print("data_len:", len(data))

sample = "To be or not to be"
sample_ids = encode(sample)

print("encoded:", sample_ids)
print("decoded:", decode(sample_ids))

vocab_size: 5000
data_len: 145812398
encoded: [2045, 309, 392, 426, 241, 309]
decoded:  To  be  or  not  to  be


In [6]:
def make_causal_mask(T: int, device=None):
    mask=torch.tril(torch.ones(T, T, device=device))
    return mask

T = 5
mask = make_causal_mask(T, device=device)
print(mask)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]], device='cuda:0')


In [7]:
class SingleHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, head_dim: int, block_size: int): #block_size:模型允许的最大长度
        super().__init__()
        self.head_dim=head_dim
        self.q= nn.Linear(n_embd,head_dim)
        self.k= nn.Linear(n_embd,head_dim)
        self.v=nn.Linear(n_embd,head_dim)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size))) #保留下三角，防止预测时看到未来的token

    def forward(self, x):
        # x: (B,T,C)
        B,T,C = x.shape
        q = self.q(x) # (B, T, head_dim)
        k = self.k(x) # (B, T, head_dim)
        v = self.v(x) # (B, T, head_dim)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)   #(T,head_dim)@(head_dim,T)->(T,T)
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf")) #注释在下面
        weight= F.softmax(att, dim=-1) #这里-1不是自动找维度，attn.shape=[B,T(query token),T(key_token)]所以输出维度应该与key_token维度一致
        out_put = weight @ v
        return out_put

# quick shape test
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
attn = SingleHeadCausalSelfAttention(n_embd=C, head_dim=8, block_size=block_size)
out = attn(x)
print("out shape:", out.shape)

out shape: torch.Size([4, 8, 8])


In [8]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.gate = nn.Linear(n_embd, n_embd, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        attn_qk = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_qk = attn_qk.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        attn_weight = F.softmax(attn_qk, dim=-1)
        attn_weight = self.dropout(attn_weight)

        attn = attn_weight @ v
        attn = attn.transpose(1, 2).contiguous().view(B, T, C)

        gate = torch.sigmoid(self.gate(attn))
        attn = self.proj(attn)
        attn = attn * gate
        attn = self.dropout(attn)
        return attn

In [9]:
class GatedFeedForward(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.0):
        super().__init__()
        hidden = 4 * n_embd
        self.fc = nn.Linear(n_embd, hidden)
        self.gate = nn.Linear(n_embd, hidden)
        self.proj = nn.Linear(hidden, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = F.gelu(self.fc(x))                 # 内容分支
        g = torch.sigmoid(self.gate(x))       # 门控分支
        x = h * g
        x = self.proj(x)
        x = self.dropout(x)
        return x


# shape test
x = torch.randn(3, 7, 32)
ff = GatedFeedForward(32)
y = ff(x)
print("y shape:", y.shape)

y shape: torch.Size([3, 7, 32])


In [10]:
class TransformerBlock(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.mha = MultiHeadCausalSelfAttention(
            n_embd=n_embd,
            n_head=n_head,
            block_size=block_size,
            dropout=dropout,
        )
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = GatedFeedForward(n_embd, dropout=dropout)

    def forward(self, x):
        x = x + self.mha(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x



# shape test
x = torch.randn(2, 16, 64)
blk = TransformerBlock(n_embd=64, n_head=4, block_size=block_size)
y = blk(x)
print("y shape:", y.shape)

y shape: torch.Size([2, 16, 64])


In [11]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size: int, block_size: int, n_embd: int = 128, n_head: int = 4, n_layer: int = 4, dropout: float = 0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)


    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss



In [12]:
@torch.no_grad()
def generate(model, idx, max_new_tokens=100, temperature=1.0, top_k=None):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        next_logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
            next_logits[next_logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(next_logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return idx
model = TinyGPT(vocab_size=vocab_size, block_size=block_size, n_embd=128, n_head=4, n_layer=4).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

for step in range(20000):
    xb, yb = get_batch(device)
    logits, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 2000 == 0:
        print("step", step, "loss", float(loss))

# save model
torch.save(model.state_dict(), "./model.pt")
print("model saved")

C:\Users\沈尚诣\AppData\Local\Temp\ipykernel_20452\2168090031.py:27: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  print("step", step, "loss", float(loss))


step 0 loss 8.665082931518555
step 2000 loss 5.0788373947143555
step 4000 loss 4.750362396240234
step 6000 loss 4.488649368286133
step 8000 loss 4.6232008934021
step 10000 loss 4.495267868041992
step 12000 loss 4.360638618469238
step 14000 loss 4.23602294921875
step 16000 loss 4.362452983856201
step 18000 loss 4.29121208190918
model saved


In [24]:
prompt = "Once upon a time"

ids = tokenizer.encode(prompt).ids
idx = torch.tensor([ids], dtype=torch.long).to(device)

out = generate(
    model,
    idx,
    max_new_tokens=100,
    temperature=0.8,
    top_k=40
)

generated_ids = out[0].tolist()
text = tokenizer.decode(generated_ids)

print(text)


ĠOn ce Ġupon Ġa Ġtime Ġin Ġher Ġc inem a Ġ, Ġwhich Ġhas Ġbeen Ġused Ġin Ġthe Ġn inet een Ġ@-@ Ġday ĠT rib e Ġ@-@ ĠF at ra Ġand ĠSh Åį g Åį Ġ. ĠIn ĠFebruary Ġ2012 Ġ, ĠH ih anna Ġsigned Ġa Ġcontract Ġwith Ġa Ġnew Ġteam Ġfor ĠD omin ia ĠR u ai Ġon ĠJanuary Ġ1 Ġ, Ġ2005 Ġ. ĠIn Ġthe Ġ2011 ĠInd epend ent Ġ, Ġshe Ġappeared Ġto Ġperform Ġin Ġa Ġmed al Ġ@-@ Ġs ession Ġof Ġthe ĠR ang er ĠS a it i Ġ, ĠOh io Ġ, Ġwhere Ġshe Ġwon Ġher Ġat Ġa Ġuniversity Ġ. ĠShe Ġwas Ġselected Ġto
